Appendix A: Introduction to PyTorch (Part 2)
A.9 Optimizing training performance with GPUs
A.9.1 PyTorch computations on GPU devices

In [ ]:
import torch  # 导入 PyTorch 核心库

print(torch.__version__)          # 打印当前安装的 PyTorch 版本号
print(torch.cuda.is_available())  # 检查当前设备是否支持 CUDA(即是否有可用的 GPU)

# 创建两个形状为 (3,) 的一维张量,默认存储在 CPU 上
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

print(tensor_1 + tensor_2)  # 两个 CPU 张量逐元素相加,结果形状仍为 (3,)

In [ ]:
# 通过 .to("cuda") 将张量从 CPU 拷贝到默认的 GPU 设备上
# 注意:若当前机器没有可用的 GPU,这一步会抛出 RuntimeError
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")

# 两个张量都在 GPU 上时可以正常相加,计算在 GPU 上完成
print(tensor_1 + tensor_2)

In [ ]:
# 将 tensor_1 移回 CPU,但 tensor_2 仍留在 GPU 上
tensor_1 = tensor_1.to("cpu")

# 此时 tensor_1 在 CPU、tensor_2 在 GPU,两者设备不一致
# PyTorch 要求参与运算的张量必须位于同一设备,否则会抛出 RuntimeError
# 这个单元格是刻意演示"跨设备运算会报错"这一现象,用于教学说明,不是需要修复的 bug
print(tensor_1 + tensor_2)

In [ ]:
# 训练数据:5 个样本,每个样本 2 个特征,形状为 (5, 2)
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

# 训练标签:5 个样本各自的类别(0 或 1),形状为 (5,)
y_train = torch.tensor([0, 0, 0, 1, 1])

# 测试数据:2 个样本,每个样本 2 个特征,形状为 (2, 2)
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

# 测试标签:形状为 (2,)
y_test = torch.tensor([0, 1])
from torch.utils.data import Dataset  # PyTorch 数据集基类


# 自定义 Dataset:把特征张量和标签张量包装成可按索引取样的数据集
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X  # 形状 (N, 2)
        self.labels = y    # 形状 (N,)

    def __getitem__(self, index):
        # 按索引取出单个样本的特征和标签
        one_x = self.features[index]  # 形状 (2,)
        one_y = self.labels[index]    # 标量
        return one_x, one_y

    def __len__(self):
        # 数据集大小 = 标签数量 N
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)  # 5 个训练样本
test_ds = ToyDataset(X_test, y_test)     # 2 个测试样本
from torch.utils.data import DataLoader

torch.manual_seed(123)  # 固定随机种子,保证 DataLoader 打乱(shuffle)结果可复现

# 训练集 DataLoader:批大小为 2,打乱样本顺序,用 1 个子进程加载数据
# drop_last=True 表示丢弃最后一个样本数不足 batch_size 的批次
# (5 个训练样本按 batch_size=2 划分,最后会剩 1 个样本,因而被丢弃)
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=1,
    drop_last=True
)

# 测试集 DataLoader:批大小为 2,不打乱顺序(评估时保持样本原始顺序即可)
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=1
)
# 一个简单的多层感知机(MLP):输入层 -> 两个隐藏层(ReLU 激活) -> 输出层
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            # 第 1 个隐藏层:输入维度 num_inputs -> 30,后接 ReLU 激活
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            # 第 2 个隐藏层:30 -> 20,后接 ReLU 激活
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            # 输出层:20 -> num_outputs,输出未做 softmax 归一化的 logits
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        # x 形状: (batch_size, num_inputs) -> logits 形状: (batch_size, num_outputs)
        logits = self.layers(x)
        return logits
import torch.nn.functional as F


torch.manual_seed(123)  # 固定随机种子,保证模型参数初始化可复现
model = NeuralNetwork(num_inputs=2, num_outputs=2)  # 2 个输入特征,2 分类输出

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # NEW
# 若检测到可用 GPU 则用 "cuda",否则退回 "cpu";实现与硬件无关的代码
model.to(device) # NEW
# 将模型的全部参数和缓冲区搬到目标设备上;nn.Module.to() 是原地操作

# Note that the book originally used the following line, but the "model =" is redundant
# model = model.to(device) # NEW
# 说明:nn.Module.to() 会原地修改模型自身并返回 self,
# 因此不需要写成 model = model.to(device),此处保留原书注释供参考

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)  # 随机梯度下降优化器,学习率 0.5

num_epochs = 3  # 训练轮数(epoch 数)

for epoch in range(num_epochs):

    model.train()  # 切换到训练模式(本例无 Dropout/BatchNorm,但这是良好实践)
    for batch_idx, (features, labels) in enumerate(train_loader):

        # 将当前批次的特征和标签搬到与模型相同的设备上,否则前向计算会因设备不一致而报错
        features, labels = features.to(device), labels.to(device) # NEW
        logits = model(features)  # 前向传播,logits 形状 (batch_size, 2)
        loss = F.cross_entropy(logits, labels) # Loss function
        # 交叉熵损失:内部会自动对 logits 做 softmax,再计算负对数似然

        optimizer.zero_grad()  # 清空上一步遗留的梯度,避免梯度累积
        loss.backward()        # 反向传播,计算各参数的梯度
        optimizer.step()       # 根据梯度更新模型参数

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation
    # 切换到评估模式;此处未展开具体评估逻辑(留空,评估函数在下一单元格实现)

In [ ]:
# 计算模型在给定 dataloader 上的分类准确率
def compute_accuracy(model, dataloader, device):

    model = model.eval()  # 切换到评估模式
    correct = 0.0          # 累计预测正确的样本数
    total_examples = 0     # 累计样本总数

    for idx, (features, labels) in enumerate(dataloader):

        # 将该批次数据搬到与模型相同的设备上
        features, labels = features.to(device), labels.to(device) # New

        with torch.no_grad():  # 评估阶段无需计算梯度,节省显存并加速
            logits = model(features)  # 形状: (batch_size, num_outputs)

        predictions = torch.argmax(logits, dim=1)  # 取每个样本 logits 最大值对应的类别索引,形状 (batch_size,)
        compare = labels == predictions            # 逐元素比较预测值与真实标签,得到布尔张量
        correct += torch.sum(compare)              # 累加本批次预测正确的样本数
        total_examples += len(compare)             # 累加本批次的样本数

    return (correct / total_examples).item()  # 返回标量准确率(Python float)
compute_accuracy(model, train_loader, device=device)  # 在训练集上计算准确率

In [ ]:
# 在测试集上计算准确率,用于评估模型的泛化效果
compute_accuracy(model, test_loader, device=device)

A.9.3 Training with multiple GPUs